# 17A — Final Cycle 24 → Cycle 25 temporal multimodal protocol

**Date:** 8 September 2026  
**Status:** Implemented protocol scaffold; exact scientific choices remain **PROPOSED, NOT FROZEN**.  
**Compute:** Python standard library; no network, model training or GPU required by this notebook.

This notebook continues **after completed Notebook 16B**, rather than resuming June extraction recovery. Its starting evidence is GitHub Issue #1, `docs/final_bigbang_cycle24_to_cycle25_execution_plan_2026-09-08.md`, and the completed 16B all-fold report. A proposed next stage is not an executed experiment.

Existing result scope: Cycle 24 chronological folds 2013, 2014 and 2015. AIA ResNet18 plus SHARP 24-hour magnetic aggregates. GOES was label lineage only. The separate physics-informed track remains 20A–20C.


In [ ]:
from pathlib import Path
import json
import sys

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents]
             if (p / "configs/aia17_metadata_stage1.json").is_file()),
            Path.home() / "solar_flare_aia")
if not (ROOT / "src/aia17_metadata_audit.py").is_file():
    raise FileNotFoundError("Install the complete AIA 17A/17B package before running this notebook.")
sys.path.insert(0, str(ROOT / "src"))
from aia17_metadata_audit import check_protocol, run_pipeline
CONFIG = json.loads((ROOT / "configs/aia17_metadata_stage1.json").read_text())
check_protocol(CONFIG)
print("ROOT:", ROOT)
print("Protocol status:", CONFIG["split_status"])
print("Training authorised:", CONFIG["training_authorised"])


## 1. Existing scientific contract

Predict a same-active-region M/X flare in **`(t, t + 48h]`** using `label_48h_final`. Ignore embedded NPZ labels and retain legacy label columns only for lineage/audit.

AIA source configuration: **94, 131, 171, 193, 211, 335 Å**, **96-minute cadence**, stored **512 × 512 × 6** tensors. These are prior specifications, not a new tensor-integrity result. The 180-second acquisition tolerance is not permission to include an image after issue time `t`.

Train-only preprocessing and validation-only checkpoint/threshold selection remain mandatory. Identifiers, quality flags, paths, labels and future-window information must never be indiscriminately included as features.


In [ ]:
print(json.dumps({k: CONFIG[k] for k in ["label", "forecast_window", "aia_channels_angstrom", "aia_source_cadence_minutes", "aia_time_tolerance_seconds"]}, indent=2))


## 2. A concrete date-role proposal — not a frozen split

The committed plan supplied alternatives. The following is a **new proposed nonoverlapping year allocation**, to be reviewed before modelling:

| Role | Proposed years |
|---|---|
| Training | 2010–2017 |
| Validation | 2018–2019 |
| Transition sensitivity, separate | 2020 |
| Independent Cycle-25 test | 2021–2025 |
| Later extension, outside the primary test | 2026 |

Freeze exact timestamp boundaries and time scales; establish the AR-group policy; purge training targets whose forecast windows cross a later split boundary. Audit the complete input-history/forecast intervals, not just the issue dates. Do not change dates or thresholds in response to test-model performance.

A candidate year containing few labelled positives is not by itself grounds for silently selecting another validation year. Follow-up catalogue coverage through `t+48h` must be established independently of the last observed flare. This notebook creates no train/test tensors.


In [ ]:
print(json.dumps(CONFIG["proposed_date_roles"], indent=2))
print("Exact split frozen?", CONFIG["split_status"] == "FROZEN")


## 3. Source roles and the 17B metadata-first stage

The curated master and 2025–2026 extension have official-label columns; the raw SHARP files are magnetic history sources and do not need classification labels. Compare the extension against the master by unique `sample_id` plus HARP, NOAA, time and label; do **not** append both blindly.

The 2010–2016 baseline manifest supplies the established labelled baseline reference. A curated target row is **not** proof of a stored image. Do not infer the full archive size from the 68,010-sample baseline.

Observed archive locations: `samples_npz/` for 2010–2024; `jsoc_2025_2026_production_v1/samples_npz/` for 2025–2026. This first audit does not download image payloads or the 1.79 GB twelve-minute SHARP file. It first profiles the smaller 96-minute history; this does not settle the final temporal-input cadence.


In [ ]:
for s in CONFIG["sources"]:
    print(s["id"], "→", s["role"], "→", s["uri"])
print("Bulk file deliberately excluded:", CONFIG["excluded_bulk_source"])


## 4. Time and feature availability

A naive `T_REC_dt` is not automatically UTC. Inspect original `T_REC` tags and construction code before reconciling SHARP with AIA and event times. Stage 1 reports representations but performs **no TAI-to-UTC conversion** and no cross-instrument time join.

For GOES/HEK history, event start before `t` does not mean its final peak class, peak flux or duration was known at `t`. Peak-derived fields require peak time at or before `t`; end-derived fields require end time at or before `t`; any reporting delay requires additional handling. Count actual observed class families before deciding which features are supported. An M/X-only catalogue cannot establish C-class quietness or replace continuous GOES XRS data.

The previously audited preferred SHARP list had **15 available columns and missing `TOTUSJH`**. Recheck the current schema, do not invent this column, and do not substitute `TOTUSJZ` for it. Neither `QUALITY` nor a history-count field is automatically a physical model feature.


## 5. Controlled modelling sequence

**17A:** protocol → **17B:** readiness → **17C:** loader canary → **18A:** cross-cycle SHARP/GOES → **18B:** temporal AIA → **18C:** full temporal fusion → **19A:** explainability, ablations and manuscript evidence.

Preserve the model ladder in the plan: SHARP; past-only GOES; SHARP+GOES; temporal AIA; temporal AIA+SHARP; full AIA+SHARP+GOES; optional HMI-map branch. Compare modalities on matching sample populations and also report coverage. Do not claim all architectures are already chosen or run.

Report TSS, HSS, ROC-AUC, PR-AUC (with its estimator explicitly named), precision, recall, F1, Brier score and confusion counts. Preserve validation thresholds in test evaluation. Add calibration/error analysis and appropriate grouped uncertainty estimates to the later protocol before final runs.


## 6. Separate PINN/PIML track

**20A:** local/CPU heat-equation PINN demonstration. **20B:** SHARP physical-proxy auxiliary-learning prototype. **20C:** figures, results and an interview-safe limitations statement. This is not a completed full solar-MHD solver and is not silently integrated into the main sprint.

## 7. Cost and persistence correction

This stage only stages exact CSV objects to a local cache and writes local audit reports. No billing change, VM creation, GPU installation, bucket update, source deletion or automatic Git push is permitted. Cloud Storage operations/transfers can still be billable.

Do not treat routine billing disablement of the archival-data project as safe hibernation. Check compute and storage costs separately, verify backups before any deletion, and use a named budgeted run with a stop condition for paid compute. The archive stays untouched. Generated reports and SQLite/cache files live **outside** the repository by default.

## 8. What freezes the protocol?

Not this notebook alone. Review 17B source/coverage/timescale evidence, record exact splits, feature availability and missingness rules, verify AR/purge checks and backup loadability, then record explicit approval. Until then: **NO TRAINING AUTHORISED**.

### Project evidence

- GitHub Issue #1 in `Watchman77/solar-flare-aia-training`.
- `docs/final_bigbang_cycle24_to_cycle25_execution_plan_2026-09-08.md`.
- `results/metrics/aia_resnet18_sharp24h_intermediate_fusion_cycle24_fullnatural_allfold_report.md`.
- `results/metrics/aia_sharp_goes_fusion_protocol.md` (3 August: preferred-feature availability and availability-time caution).
- `docs/separate_pinn_piml_portfolio_track_2026-09-08.md`.
- User-provided Cloud Shell source/header/backup probe, 8 September 2026.


In [ ]:
print("17A protocol scaffold checked.")
print("Split status:", CONFIG["split_status"])
print("Training authorised:", False)
print("Next: 17B stage 1 metadata content audit; remaining stages stay open.")


## Adopted evaluation standard — UQ and calibration addendum

**Performance → Calibration → Uncertainty → Robustness → Explainability → Statistical significance.**

Read `docs/TRUSTWORTHY_RESEARCH_STANDARD.md` and `docs/AIA_UQ_CALIBRATION_PROTOCOL.md` before freezing the final model protocol. Configuration: `configs/aia_uq_calibration_protocol.json`.

Three different questions: per-prediction model uncertainty; dependency-aware 95% CIs for metrics; probability calibration. Candidate MC-dropout/deep-ensemble sizes are provisional, not validated settings. Calibrators and thresholds use development data only; final test labels never fit either. Resample audited AR groups, not automatically independent rows. Retain raw probabilities, scores/logits where available and calibrated outputs. Historical later-fold checkpoints must not leak into an earlier fold through an ensemble.

This cell adds requirements, not completed uncertainty results. Keep the separate PINN/PIML track separate. Existing split/date/purge decisions remain unfrozen.
